# STIX Generator — Phase 1 Walkthrough

This notebook runs the same pipeline as `python -m stix_generator.pipeline`, one stage at a time, so you can inspect and tinker with the intermediate results.

**It does not reimplement anything.** Every cell below just calls functions that live in the `stix_generator/` package — if you change the prompt, schema, or construction logic, edit the files there and re-run this notebook; don't copy logic into cells.

**Before running:** in the kernel picker (top right), select **"STIX Generator (.venv)"**. If you don't see it, see the setup notes from the assistant.

**Cost/time note:** the extraction cell (Step 2) is the only one that calls the Claude API. It costs a small amount and takes ~10-30 seconds. Everything else is free and instant, so feel free to re-run construction/validation cells repeatedly once you have an `extraction` object.

In [1]:
# Setup: confirm we're running from the project root, load the API key, import pipeline pieces.
from pathlib import Path
import json

print("Working directory:", Path.cwd())
assert (Path.cwd() / "stix_generator").exists(), (
    "Can't find the stix_generator package from here. "
    "Launch Jupyter Lab from the 'STIX Generator' project folder, "
    "or move/open this notebook there."
)

from dotenv import load_dotenv
load_dotenv()

from stix_generator.ingestion.loader import load_report
from stix_generator.extraction.extractor import extract, DEFAULT_MODEL
from stix_generator.construction.builder import build_bundle
from stix_generator.validation.validator import validate_bundle

print("Ready. Default model:", DEFAULT_MODEL)

Working directory: C:\Users\matth\Documents\STIX_Generator
Ready. Default model: claude-sonnet-5


## 1. Load a report

`load_report` handles `.pdf`, `.txt`, and `.md` transparently. Point `report_path` at any file in `data/reports/`.

In [2]:
report_path = Path("data/reports/autonomous-ai-cyber-attack-campaign.pdf")

report_text = load_report(report_path)
print(f"{len(report_text):,} characters loaded\n")
print(report_text[:600], "...")

21,259 characters loaded

Chinese-Speaking Threat Actor Harnesses AI
Models for Autonomous Cyberattacks
Executive Summary
Independently enumerating targets and their vulnerabilities using FOFA
Sourcing exploit tools
Initiating attacks without human intervention
Unit 42 identified an AI-enabled autonomous hacking campaign carried out by a Chinese-speaking
threat actor. They targeted infrastructure using seven vulnerabilities, combining autonomous AI-driven
enumeration with manual exploitation that achieved confirmed impact.
The actor, operating under the aliases knaithe and KnYuan, leveraged DeepSeek, via the Hermes
Age ...


## 2. Extract entities, observables, and relationships (calls Claude)

This is the only LLM step. It returns an `ExtractionResult` — the intermediate representation described in `stix_generator/extraction/schema.py` — **not** STIX yet. No IDs, no schema types, just what the model read out of the text.

In [3]:
extraction = extract(report_text)

print(f"{len(extraction.entities)} entities")
print(f"{len(extraction.observables)} observables")
print(f"{len(extraction.relationships)} relationships")

34 entities
5 observables
56 relationships


### Inspect the raw extraction

Worth reading closely — this is where you'll catch misreadings before they get baked into STIX objects (e.g. something classified as `malware` that's really a dual-use `tool`).

In [4]:
for e in extraction.entities:
    aliases = f" (aka {', '.join(e.aliases)})" if e.aliases else ""
    print(f"[{e.local_id}] {e.type}: {e.name}{aliases}")

print()
for o in extraction.observables:
    print(f"[{o.local_id}] {o.observable_type}: {o.value}")

print()
for r in extraction.relationships:
    print(f"{r.source_local_id} --{r.relationship_type}--> {r.target_local_id}")

[TA1] threat-actor: knaithe (aka KnYuan)
[MAL1] malware: DeepSeek
[TOOL1] tool: Hermes Agent
[TOOL2] tool: Claude Code
[TOOL3] tool: Codex
[TOOL4] tool: Qwen Code
[MAL2] malware: Qwen
[MAL3] malware: GLM
[MAL4] malware: Kimi
[MAL5] malware: MiniMax
[TOOL5] tool: FOFA
[TOOL6] tool: FofaMap-Platinum-Full-Expert MCP server (aka FofaMap)
[TOOL7] tool: fofoapi.py
[TOOL8] tool: 1DayNews
[TOOL9] tool: Nuclei
[TOOL10] tool: Telegram
[VULN1] vulnerability: CVE-2026-33017
[VULN2] vulnerability: CVE-2026-21858
[VULN3] vulnerability: CVE-2025-68613
[VULN4] vulnerability: CVE-2026-3055
[VULN5] vulnerability: CVE-2026-39987
[VULN6] vulnerability: CVE-2026-34486
[VULN7] vulnerability: CVE-2026-33824
[VULN8] vulnerability: CVE-2026-0300
[AP1] attack-pattern: Exploit Public-Facing Application
[AP2] attack-pattern: Command and Scripting Interpreter
[AP3] attack-pattern: Data from Information Repositories / Exfiltration
[AP4] attack-pattern: Active Scanning
[AP5] attack-pattern: Proxy: Multi-hop Proxy
[C

### (Optional) Tinker: hand-edit an entity before building

`extraction` is a live Python object — you can change it before Step 3 uses it. Example: reclassify an entity the model called `malware` as a `tool` instead. Find the `local_id` from the printout above, then edit and re-run.

This cell is inert by default (the `if False:` guard) — flip it to `True` to actually apply the edit, or write your own.

In [5]:
def find_entity(extraction, local_id):
    return next(e for e in extraction.entities if e.local_id == local_id)

if False:
    entity = find_entity(extraction, "MAL1")  # replace with the local_id you want to change
    entity.type = "tool"
    print("Updated:", entity)

## 3. Build the STIX bundle (deterministic — no LLM involved)

`build_bundle` turns the IR into real STIX 2.1 objects: assigns `type--uuid` IDs, fills in spec-required fields, builds indicator patterns for observables, and resolves relationships by `local_id`. Any relationship whose `local_id` doesn't resolve gets dropped with a warning rather than silently producing a broken bundle.

In [6]:
bundle, warnings = build_bundle(extraction)

print(f"{len(bundle.objects)} STIX objects built\n")
for w in warnings:
    print("WARNING:", w)

95 STIX objects built



## 4. Validate against the STIX 2.1 schema

In [7]:
bundle_json = bundle.serialize(pretty=True)
validation = validate_bundle(bundle_json)

print("VALID" if validation["is_valid"] else "INVALID")
for e in validation["errors"]:
    print("ERROR:", e)
for w in validation["warnings"]:
    print("SCHEMA WARNING:", w)

VALID
SCHEMA WARNING: threat-actor--38651226-6b88-49be-89a9-22bc4c4841a8: {111} Open vocabulary value 'exploit operator' should be all lowercase and use hyphens instead of spaces or underscores as word separators.
SCHEMA WARNING: threat-actor--38651226-6b88-49be-89a9-22bc4c4841a8: {211} The value contained in primary_motivation is permitted, but is not in the attack-motivation-ov vocabulary.
SCHEMA WARNING: threat-actor--38651226-6b88-49be-89a9-22bc4c4841a8: {220} The value contained in roles is permitted, but is not in the threat-actor-role-ov vocabulary.
SCHEMA WARNING: malware--9c97c415-e0ac-47a2-82d3-4e0d51fbe0b6: {216} The value contained in malware_types is permitted, but is not in the malware-type-ov vocabulary.
SCHEMA WARNING: tool--3adaf00e-d884-492c-b02f-e4d937a5de42: {222} The value contained in tool_types is permitted, but is not in the tool-type-ov vocabulary.
SCHEMA WARNING: tool--445f42e2-955e-4e21-b567-26189ca2e3cc: {222} The value contained in tool_types is permitted, 

## 5. Browse the bundle as a graph

STIX objects never nest — relationships are separate objects pointing at other objects by ID. This helper resolves those IDs back to readable names so you can read the graph as `A --relationship--> B`.

In [8]:
def describe_bundle(bundle_json: str) -> None:
    objects = json.loads(bundle_json)["objects"]
    by_id = {obj["id"]: obj for obj in objects}

    def label(obj: dict) -> str:
        return obj.get("name") or obj.get("id", "?")

    print("Objects:")
    for obj in objects:
        if obj["type"] != "relationship":
            print(f"  [{obj['type']}] {label(obj)}")

    print("\nRelationships:")
    for obj in objects:
        if obj["type"] == "relationship":
            src = by_id.get(obj["source_ref"], {})
            tgt = by_id.get(obj["target_ref"], {})
            print(f"  {label(src)} --{obj['relationship_type']}--> {label(tgt)}")

describe_bundle(bundle_json)

Objects:

  [threat-actor] knaithe (KnYuan)

  [malware] DeepSeek

  [tool] Hermes Agent

  [tool] Claude Code

...

  Claude Code --communicates-with--> code.newcli.com/ultra

  DeepSeek --communicates-with--> api.deepseek.com

  Qwen Code --communicates-with--> dashscope.aliyuncs.com

  code.newcli.com --hosts--> code.newcli.com/codex/v1

  code.newcli.com --hosts--> code.newcli.com/ultra

## 6. Save the bundle

Same output location the CLI uses, so you can diff notebook output against a CLI run.

In [9]:
output_path = Path("data/output") / f"{report_path.stem}.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(bundle_json, encoding="utf-8")
print("Saved to", output_path)

Saved to

data\output\autonomous-ai-cyber-attack-campaign.json

## Next steps

- Point `report_path` in Step 1 at a different file in `data/reports/` (drop a new PDF there and rerun).
- Edit `stix_generator/extraction/prompts.py` and re-run from Step 2 to see how prompt changes affect extraction — no need to touch this notebook.
- Use the tinker cell in Step 2 to hand-correct extraction output — this is exactly the workflow Phase 2 (building a gold-standard test corpus) will use.